In [19]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain
from collections import Counter

import igraph as ig
import matplotlib.pyplot as plt
import seaborn.objects as so
import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/economicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [20]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect()
        self.db.sql(f"ATTACH IF NOT EXISTS '{MY_DATABASE_FILE}' AS project")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        for tab in ['project.edge_list_combined']: 
            self.db.sql(f"DROP TABLE IF EXISTS {tab}")

        # with contextlib.suppress(Exception):
        #     self.db.create_function('normalise_name', 
        #                                 normalise_name, 
        #                                 return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
        #                                 exception_handling='return_null',
        #                                 null_handling='special',
        #                                 side_effects=True
        #                             )
                        
        print(self.db.sql("SHOW ALL TABLES").df())
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname
            

### This class constructs the pageRank for sources linked by citation counts

-  build the edge list of journal (citer) -> journal (cited)  
-  construct an iGraph from the edge list  
-  run pageRank  
-  for each author, calculate the number of ciations and the number of citations weighted by the pageRank of the citing journal



In [21]:
class PageRanks(SetUp):

    def __init__(self):
        super().__init__()
        return

    def vertex_labels(self):
        sql = """
            SELECT DISTINCT source_id AS id,
                    source_name AS name
                FROM project.works
            UNION
            SELECT DISTINCT institution_id AS id,
                    institution_name AS name
                from project.authorships
            """
        df = self.db.sql(sql).df()
        self.vertex_labels = dict(zip(df.id, df.name))
        return
    
    def make_source_edge_list(self):
        # SQL code to assemble the edge list with the count of work-work citations as weights
        sql = """
            CREATE OR REPLACE TABLE project.edge_list_sources AS
                -- Edge list for Pagerank via SOURCES
                WITH
                -- Step 1: Expand referenced works
                citer_cited_CTE AS (
                    SELECT 
                        c.work_id AS citer_id,
                        unnest AS cited_id
                    FROM project.cited c,
                        UNNEST(c.referenced_works)
                    WHERE unnest IN (SELECT work_id FROM project.cited)
                ),
                -- Step 2: Attach source IDs to both citer and cited works
                sources_citer_cited_CTE AS (
                    SELECT
                        cc.citer_id,
                        w1.source_id AS citer_unit,
                        cc.cited_id,
                        w2.source_id AS cited_unit
                    FROM citer_cited_CTE cc
                    INNER JOIN project.works w1 ON cc.citer_id = w1.work_id
                    INNER JOIN project.works w2 ON cc.cited_id = w2.work_id
                    WHERE citer_unit != cited_unit
                )
                -- Step 3: Aggregate edge weights
                SELECT
                    DISTINCT citer_unit,
                            cited_unit,
                            COUNT(*) AS weights
                FROM sources_citer_cited_CTE
                GROUP BY citer_unit, cited_unit
        """
        self.db.sql(sql)
        return
    
    def make_source_adjaceny_matrix(self):
        sql = """  
            WITH nodes AS (
                SELECT DISTINCT citer_unit AS node FROM project.edge_list_sources
                UNION
                SELECT DISTINCT cited_unit AS node FROM project.edge_list_sources
            )
            SELECT
                n1.node AS citer_unit,
                n2.node AS cited_unit,
                COALESCE(e.weights, 0) AS weight
            FROM nodes n1
            CROSS JOIN nodes n2
            LEFT JOIN project.edge_list_sources e
                ON n1.node = e.citer_unit AND n2.node = e.cited_unit
            ORDER BY n1.node, n2.node
            """
        self.db.sql(sql).show()
        return
    
    def make_institution_edge_list(self):
        # SQL code to assemble the edge list with the count of work-work citations as weights
        sql = """
                CREATE OR REPLACE TABLE project.edge_list_institutions AS
                    -- Edge list for Pagerank via INSTITUTIONS, with unit weight per (citer_id, cited_id)
                    WITH
                    -- Step 1: Expand referenced works and deduplicate (citer_id, cited_id)
                    unique_citations AS (
                        SELECT DISTINCT work_id AS citer_id,
                                        unnest(referenced_works) AS cited_id
                        FROM project.cited
                    ),
                    -- Step 2: Attach institution IDs to both citer and cited works
                    institutions_citer_cited AS (
                        SELECT
                            uc.citer_id,
                            w1.institution_id AS citer_unit,
                            uc.cited_id,
                            w2.institution_id AS cited_unit
                        FROM unique_citations uc
                        INNER JOIN project.authorships w1 ON uc.citer_id = w1.work_id
                        INNER JOIN project.authorships w2 ON uc.cited_id = w2.work_id
                        WHERE w1.institution_id IS NOT NULL AND w2.institution_id IS NOT NULL
                    )
                    -- Step 3: Each (citer_id, cited_id) pair can generate multiple (citer_unit, cited_unit) links, each with unit weight
                    SELECT
                        citer_unit,
                        cited_unit,
                        COUNT(*) AS weights
                    FROM institutions_citer_cited
                    WHERE citer_unit != cited_unit
                    GROUP BY citer_unit, cited_unit
                    HAVING COUNT(*) <= 150
        """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.edge_list_institutions").show()
        return

    def make_combined_edge_list(self):
        # SQL code to merge journal and institution edge lists
        sql = """
            CREATE OR REPLACE TABLE project.edge_list_both AS
                SELECT citer_unit, cited_unit, weights FROM project.edge_list_sources
                UNION 
                SELECT citer_unit, cited_unit, weights FROM project.edge_list_institutions
            """
        self.db.sql(sql)     
        return

    def _extract_edge_list(self):
        kind = self.kind
        df = self.db.sql(f"SELECT citer_unit, cited_unit, weights FROM project.edge_list_{kind}").df().\
            sort_values('weights', ascending=False).reset_index(drop=True)
        return df
    
    def construct_graph(self, kind=None):
        self.kind = kind
        df_edges = self._extract_edge_list()    
        vertices = set(df_edges.citer_unit.tolist() + df_edges.cited_unit.tolist())
        vs = pd.DataFrame(data=list(vertices), columns=['id'])
        vs['label'] = [self.vertex_labels.get(id) for id in vs.id]
        print(f'{vs['id'].nunique() = } {df_edges.shape = }\n{df_edges.head()}\n{df_edges.tail()}')
        print(f'GRAPH NODES - Vertex count for {self.kind = } {vs.shape = }\n{vs.head()}')
        self.g = ig.Graph.DataFrame(df_edges, directed=True, use_vids=False, vertices=vs)
        summary = ig.summary(self.g, verbosity=1, width=256, edge_list_format='auto', max_rows=2, print_graph_attributes=True, 
                                          print_vertex_attributes=True, print_edge_attributes=True, full=False)
        print(f'*** SUMMARY OF self.g\n{summary}')
        return
    
    def run_pagerank(self):
        print('pageRank')
        damping = 0.5  # if self.kind == 'both' else 0.85
        print(f'>> RUN pagerank with {damping = } for {self.kind = }')
        ranks = self.g.pagerank(damping=damping)
        print(f'>> CHECK pageRank  - should sum to unity {sum(ranks) = } then scaled to a sum of 100 as in "eigenfactor" score')
        sum_ranks = sum(ranks)
        scaled_ranks = [100.0 * r / sum_ranks for r in ranks]
        pagerank = pd.DataFrame({
                                'pageRank': scaled_ranks,
                                'citer': self.g.vs['name'],
                                'label': self.g.vs['label'],
                                'in_degree': self.g.strength(mode='in', weights='weights'),
                                'out_degree': self.g.strength(mode='out', weights='weights')
                                }).sort_values('pageRank', ascending=False)
        pagerank['influence'] = pagerank['pageRank'] / pagerank['out_degree'].replace(0, pd.NA)
        self.db.sql(f"CREATE OR REPLACE TABLE project.pagerank_{self.kind} AS SELECT * FROM pagerank ORDER BY influence DESC")
        return
    
    def report_pagerank(self):
        kind = self.kind
        df = (
            self.db.sql(f"SELECT * FROM project.pagerank_{kind}")
            .df()
            .reset_index(drop=True)
        )
        print(f"Pagerank for {kind!r}: {df.shape}\n{df.head()}")
        print(f"Sum of out_degrees: {df['out_degree'].sum()}")
        print(f"Sum of pageranks: {df['pageRank'].sum()}")
        sum_weights = self.db.sql(f"SELECT SUM(weights) AS total_weights FROM project.edge_list_{kind}").df().iloc[0, 0]
        print(f"Sum of edge weights: {sum_weights}")
        return
    
    def run_reputation_both(self):
        df = self.db.sql("SELECT * FROM project.pagerank_both").df()
        df_s = self.db.sql("SELECT * FROM project.pagerank_sources").df()
        print(f'{df_s.shape = }\n{df_s.head()}')
        df_i = self.db.sql("SELECT * FROM project.pagerank_institutions").df()
        print(f'{df_i.shape = }\n{df_i.head()}')
        dd = dict(zip(df_s.citer, df_s.influence)) | dict(zip(df_i.citer, df_i.influence))
        df['influence'] = [dd.get(item) for item in df.citer]
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE project.pagerank_both AS (SELECT * FROM df)")
        return

In [22]:

class WeightedCitationCounts(SetUp):

    def __init__(self):
         super().__init__()
         return
    
    def make_weighted_citations(self):

        for kind in ['sources', 'institutions', 'both']:
            sql = f"""
                CREATE OR REPLACE TABLE project.weighted_citations_{kind} AS
                    WITH 
                    item_cte AS
                        (SELECT DISTINCT citer_item
                        FROM project.edge_list_combined
                        ), -- cte to filter combined list
                    source_cte AS
                        (SELECT citer_item AS source_id
                        FROM item_cte
                        WHERE contains(citer_item, '/S') = true
                        ), -- cte to filter sources from combined list
                    institution_cte AS
                        (SELECT citer_item AS institution_id
                        FROM item_cte
                        WHERE contains(citer_item, '/I') = true
                        ), -- cte to filter institutions     p = Plotters()
                    works_sources_institutions AS
                        (
                        SELECT DISTINCT w.work_id, w.source_id, a.institution_id
                        FROM project.works w
                        LEFT JOIN project.authorships a
                        USING (work_id)
                        LEFT JOIN source_cte s
                        ON s.source_id = w.source_id
                        LEFT JOIN institution_cte i
                        ON i.institution_id = a.institution_id
                        WHERE a.institution_id NOT NULL AND w.source_id NOT NULL
                        ORDER BY w.source_id, a.institution_id
                        ), -- cte to build a table of the sources (one-to-one) and institutions (one-to-many) for each work 
                    work_reputation_sources AS
                        (
                        SELECT DISTINCT w.work_id, influence AS reputation -- pageRank AS reputation
                        FROM works_sources_institutions w
                        LEFT JOIN pagerank_sources p
                        ON w.source_id = p.citer
                        ), -- cte to attach source-only pageranks to works
                    work_reputation_institutions AS
                        (
                        SELECT DISTINCT w.work_id, influence AS reputation -- sum(pageRank)/count(pagerank) AS reputation
                        FROM works_sources_institutions w
                        LEFT JOIN pagerank_institutions p
                        ON w.institution_id = p.citer
                        GROUP BY ALL
                        ), -- cte to attach institution-only averaged pagerank to works
                    work_reputation_both AS
                        (SELECT sub.work_id, 0.5*(sub.reputation + p1.influence) AS reputation -- pageRank
                        FROM
                            (
                                SELECT w.work_id, w.source_id, influence AS reputation -- sum(pageRank)/count(pageRank) AS reputation
                                FROM works_sources_institutions w
                                LEFT JOIN pagerank_both p
                                ON w.institution_id = p.citer
                                GROUP BY ALL
                            ) sub
                            LEFT JOIN pagerank_both p1
                            ON sub.source_id = p1.citer
                        ), -- cte to attach averaged (source and average institution) pagerank to works
                    citer_cited AS
                        (
                        SELECT work_id AS citer_id, unnest(referenced_works) AS cited_id
                        FROM cited
                        ), -- cte to make citer-cited relation
                    citers_authors AS
                        (
                        SELECT DISTINCT author_id, c.citer_id
                        FROM project.authorships a
                        LEFT JOIN citer_cited c
                        ON a.work_id = c.cited_id
                        )  -- cte to build a list of citer works for each author 
                    
                    --SELECT * FROM item_cte
                    -- SELECT * FROM works_sources_institutions
                    -- SELECT * FROM citer_cited
                    -- SELECT * FROM citers_authors
                    -- SELECT * FROM work_pagerank_sources
                    -- SELECT * FROM work_pagerank_institutions
                    -- SELECT * FROM work_pagerank_both

                    SELECT author_id, count(DISTINCT citer_id) AS citer_count, sum(reputation) AS citer_count_weighted
                    FROM
                        (SELECT c.citer_id, c.author_id, reputation
                        FROM citers_authors c
                        LEFT JOIN work_reputation_{kind} w
                        ON c.citer_id = w.work_id
                        WHERE reputation NOT NULL)
                    GROUP BY ALL
                    ORDER BY citer_count DESC

            """
            self.db.sql(sql)
            self.db.sql(f"SELECT * FROM project.weighted_citations_{kind} ORDER BY citer_count DESC").show()
        return



In [23]:
class Plotters(SetUp):

    def __init__(self):
        super().__init__()
        return

    def plot_pagerank(self):
        hold = []
        for kind in ['sources', 'institutions', 'both']:
            temp = self.db.sql(f"SELECT * FROM pagerank_{kind}").df().sort_values('pageRank', ascending=True).reset_index(drop=True).reset_index(drop=False)
            temp['panel'] = kind
            hold.append(temp)
        df = pd.concat(hold, axis=0)
        df = df.melt(id_vars=['out_degree', 'panel'], value_vars=['pageRank', 'influence'], var_name='measure', value_name='Value')
        print(f'{df.shape = }\n{df.head()}')
        g = sns.relplot(df, x='out_degree', y='Value', hue='measure', col='panel', kind='scatter')
        for ax in g.axes.flat:
            ax.set_xlim((10, 10000))
            ax.set_ylim((0.01, 10))
            ax.set_xscale('log')
            ax.set_yscale('log')
        plt.suptitle("PageRank and Influence versus out-degree (i.e. emitted citations)", fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
        return

    def plot_model(self):

        df = self.summary[['author_id', 'citations_endogenous', 
                           'reputation_sources', 'reputation_institutions', 'reputation_both', 
                           'hca_endogenous', 'hca_total', '2yr_mean_citedness', 'h_index', 'group']]
        print(f'{df.shape = }\n{df.head()}')
        df['pointsize'] = [0.01 if s == 'X' else 5.0 if s == 'C' else 15 for s in df['group']]
        df = df.sort_values(['group', 'pointsize'], ascending=[False, True])
        hold = []
        for kind in ['sources', 'institutions', 'both']:
            temp = df.copy()
            temp['reputation'] = temp[f'reputation_{kind}']
            temp['panel'] = kind
            hold.append(temp)
        df_in = pd.concat(hold, axis=0)
        self._plot_reputations(df_in=df_in)
        self._plot_hca(df_in=df_in)

        # df = df[df.group != 'X']
        # df = df[['citer_count_weighted', 'citer_count', 'ratio', 'author_name', 'group']].sort_values('ratio', ascending=False).reset_index(drop=True)
        # df.to_csv(f'../DATA/weighted_citations_{kind}.csv')

        return

    def _plot_reputations(self, df_in=None):

        print(df_in.info())
        df = df_in.melt(id_vars=['panel', 'author_id', 'citations_endogenous', 'group', 'pointsize'], 
                        value_vars=['reputation'], 
                        value_name='Value', 
                        var_name='measure')
        print(f'{df.shape = }\n{df.head()}')      
        g = sns.relplot(df, x='citations_endogenous', y='Value', hue='group', size='pointsize', col='panel', kind='scatter', alpha=0.5)
        for ax in g.axes.flat:
            ax.set_xscale('log')
            ax.set_yscale('log')
        g.set(xlabel='Endogenous citation count per author', ylabel="Author's reputation score")
        plt.suptitle("Reputation versus citations", fontsize=16)
        sns.move_legend(g, 'upper right')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
        return

    def _plot_hca(self, df_in=None):

        for kind in ['hca_total', 'hca_endogenous', 'h_index', '2yr_mean_citedness']:

            df = df_in.melt(id_vars=['panel', 'author_id', kind, 'group', 'pointsize'], 
                            value_vars=['reputation'], 
                            value_name='Value', 
                            var_name='measure')
            print(f'{df.shape = }\n{df.head()}')    
            g = sns.relplot(df, x=kind, y='Value', hue='group', size='pointsize', col='panel', kind='scatter', alpha=0.5)
            for ax in g.axes.flat:
                ax.set_xscale('log')
                ax.set_yscale('log')
            g.set(xlabel=f"Author's {kind.replace('_', ' ').upper()}", ylabel="Author's reputation score")
            plt.suptitle(f"{kind.replace('_', ' ').upper()} versus Reputation", fontsize=16)
            sns.move_legend(g, 'upper right')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        return
        
    def extract_data(self):

        self.summary = self.db.sql("SELECT * FROM project.citation_summary").df()\
            [['author_id', 'author_name', 'works_count_endogenous', 'citations_endogenous', 'hca_total', 
             'hca_endogenous', 'works_count_total', 'cited_by_count', '2yr_mean_citedness', 'h_index']]

        for kind in ['sources', 'institutions', 'both']:
            temp = self.db.sql(f"SELECT * FROM project.weighted_citations_{kind}").df().sort_values('citer_count_weighted', ascending=False)
            temp['reputation'] = temp.citer_count_weighted/temp.citer_count
            dd = dict(zip(temp.author_id, temp.reputation))
            self.summary[f'reputation_{kind}'] = [dd.get(a) for a in self.summary.author_id]

        sample = self.db.sql("SELECT * FROM project.sample_names").df()
        dd_group = dict(zip(sample.author_id, sample.Group))
        self.summary['group'] = [dd_group.get(aid, 'X') for aid in self.summary.author_id]
        
        summary = self.summary.rename(columns={"2yr_mean_citedness": "citedness"})
        print(f'{self.summary.shape = }\n{self.summary.head()}')
        self.db.sql("SELECT * FROM summary").show()
        self.db.sql("CREATE OR REPLACE TABLE project.summary AS (SELECT * FROM summary)")
        return


In [24]:
def main():

    pr = PageRanks()
    pr.vertex_labels()
    pr.make_source_adjaceny_matrix()
    # pr.make_source_edge_list()
    # pr.make_institution_edge_list()
    # pr.make_combined_edge_list()
    # for kind in ['sources']: #, 'institutions']: #, 'both']:
    #     pr.construct_graph(kind=kind)
    #     pr.run_pagerank()
    #     pr.report_pagerank()
    # pr.run_reputation_both()

    # cn = WeightedCitationCounts()
    # cn.make_weighted_citations()

    # p = Plotters()
    # p.extract_data()
    # p.plot_pagerank()
    # p.plot_model()

In [25]:
if __name__ == "__main__":
    main()
    print("DONE!")

   database schema                             name                                                                                                                                     column_names                                                                                                                                     column_types  temporary
0   project   main              author_works_counts                                                                     [author_id, author_name, works_count_endogenous, works_count_total, h_index]                                                                                                       [VARCHAR, VARCHAR, BIGINT, BIGINT, BIGINT]      False
1   project   main                          authors  [author_id, orcid, author_name, display_name_alternatives, works_count, cited_by_count, 2yr_mean_citedness, h_index, i10_index, first, middl...                               [VARCHAR, VARCHAR, VARCHAR, VARCHAR[], BIGINT, BIGINT, DOUBLE, BIGI